# WSJ 2027 - Gruppindelning Rundresa

Assign confirmed rundresa deltagare into groups of exactly 36.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — participants should live close to each other (Hilbert curve sort)
3. **Friend wish** — at least ONE of friend_1/friend_2 in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex should be as evenly spread as possible

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 1614 participants
Confirmed: 1545, Unconfirmed: 34, Cancelled: 35
Total confirmed participants: 1544
Skipped: 34 unconfirmed, 1 wrong age/no DOB

By category:
category
deltagare    1296
ist           226
cmt            22

By travel type:
travel
rundresa      1075
direktresa     290
egen_resa      157
other           22

Friend wishes:
  With friend 1 member no: 885
  With friend 2 member no: 526
  With friend 1 name (text): 73
  With friend 2 name (text): 49

Skipped participants:
  DELTAGARE wrong age: Fredrik Berg born 1979-04-16 (age 48)


In [2]:
GROUP_SIZE = 36

# Filter to rundresa deltagare only
df_rundresa = df[df['travel'] == 'rundresa'].copy().reset_index(drop=True)
print(f"=== Rundresa deltagare ===")
print(f"Participants: {len(df_rundresa)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_rundresa)
df_sorted = u.add_hilbert_index(df_rundresa)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Rundresa deltagare ===
Participants: 1075
With coordinates: 1034
Without coordinates (Sweden centroid): 41

Groups: 29 x 36 + 1 x 31 = 30 total

By region:
region
Region Stockholm    347
Region Södra        253
Region Västra       193
Region Norr/Mitt    141
Region Östra         75
                      3

By age:
age
14    242
15    303
16    240
17    221
18     28
19     18
20      3
21     10
22      3
23      3
24      1
25      3

By sex:
sex
Kvinna    558
Man       508
Annat       7
Okänt       2


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 75
Matched & applied: 35
Generic wishes (not a person): 3
Unresolved (friend not in project): 37

Matched:
  Adelia Vallin -> Leo Norstedt (S:ta Maria Scoutkår) [rundresa] via fuzzy(0.96)
  Albin Åkesson -> Malte Lindsjö (Jonstorps Kustscoutkår) [rundresa] via fuzzy(0.78)
  Alex Liljerot Priftis -> Vera Tollwé (Årsta Scoutkår) [rundresa] via fuzzy(0.86)
  Alma Stössel Weinryb -> Lotten Hellman (Södermalms Scoutkår) [rundresa] via exact
  Alma Stössel Weinryb -> Alfons Ekelin Sintorn (Södermalms Scoutkår) [rundresa] via fuzzy(0.76)+kar
  Charlie Lindberg -> Axel Lindroth (Saltsjö-Boo Scoutkår) [rundresa] via fuzzy(0.96)
  Dag Wikström -> Fjodor Yermshuk (Roslags Näsby Scoutkår) [rundresa] via fuzzy(0.97)+kar
  Ebba Stoor -> Carl-Johan Samils (Järlinden Scoutkår) [rundresa] via exact
  Elsa Ekdahl -> Sebastian Kardin (Equmenia Scout) [rundresa] via exact
  Elsie Stenfeldt -> Hugo Johannessen (Tuve Scoutkår) [rundresa] via exact
  Emilio

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 1075
Groups: 29 x 36 + 1 x 31 = 30 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 535/643
  Kar violations: 133
  Avg geo spread: 0.4064

=== Phase 2: Fix friend wishes ===
  Swaps: 68
  Friend satisfaction: 636/643
  Kar violations: 115
  Avg geo spread: 1.1463

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 107
  Kar violations: 0
  Friend satisfaction: 519/643
  Avg geo spread: 1.1676

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 75
  Friend satisfaction: 630/643
  Kar violations: 0
  Avg geo spread: 1.3081

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 326
  Diversity score: 95.78 -> 96.57
  Avg geo spread: 1.3081 -> 1.4381

=== FINAL RESULTS ===
Groups: 29 x 36 + 1 x 31
Friend satisfaction: 632/643 (98%)
Kar violations: 0
Total swaps: 576
Diversity: 96.57
Avg geo spread: 1.4381


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 27/28      8   18/18/0   9   8   9   8        23    11
    2      36 23/23      8   15/21/0   9  12   6   8        17    12
    3      36 32/32      8   16/20/0  12  11   7   5        15     9
    4      36 24/24      8   11/25/0  12  10   6   4        36    16
    5      36 17/17      7   14/22/0  15   5   7   8        44    16
    6      36 13/14      6   22/14/0   6   8   8  10        63    25
    7      36 18/18      6   17/18/0  11  10   5   7        56    17
    8      36 22/22      6   15/21/0   9   8   7  11       108    19
    9      36 25/25      7   12/24/0   7   9  11   7        40    12
   10      36 21/22      8   14/21/1   9  16   5   6        38    13
   11      36 23/24      8   17/19/0   6   9  10  10        53    16
   12      36 20/21      5   12/24/0   4   7  11  12        90    18
   13      36 23/24  

In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_rundresa')

Saved 1075 participants to /config/notebooks/wsj27/wsj27_rundresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_rundresa_grupper.json

CSV preview (first 10 rows):
 group                name member_no  age    sex                   kar                       district           region     friend_1 friend_2       lat       lng
     1       Elsa Drakarve   3334150   17 Kvinna        Dalby Scoutkår     Södra Skånes Scoutdistrikt     Region Södra      3335276  3386856 55.664664 13.346159
     1         Ivan Valind   3412530   17    Man        Dalby Scoutkår     Södra Skånes Scoutdistrikt     Region Södra      3386856  3328923 55.664664 13.346159
     1             Lea Asp   3335276   17 Kvinna        Dalby Scoutkår     Södra Skånes Scoutdistrikt     Region Södra      3334150  3328923 55.664664 13.346159
     1      Melker Persson   3386856   17    Man        Dalby Scoutkår     Södra Skånes Scoutdistrikt     Region Södra      3334150  3412530 55.664664 13.346159
     1      

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_rundresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Rundresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Friend arcs: 598 (523 satisfied, 75 unsatisfied)
Group arcs: 18735 connections across 30 groups
Saved group map: /config/notebooks/wsj27/wsj_rundresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_rundresa_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_rundresa_grupper.json
  Map:  /config/notebooks/wsj27/wsj_rundresa_karta.html
